# PyTorch로 배우는 행렬 기초 실습

## 코드 목적

이 코드는 행렬의 기본 개념부터 행렬 연산, 전치행렬, 분할행렬, 선형연립방정식, 역행렬, 영공간까지를 PyTorch 텐서 연산으로 직접 확인하는 것이 목적입니다. 머신러닝과 딥러닝에서는 데이터, 가중치, 입력값, 출력값이 대부분 행렬 또는 텐서 형태로 표현됩니다. 따라서 행렬 연산을 이해하면 딥러닝 모델 내부에서 `입력 × 가중치 + 편향` 계산이 어떻게 수행되는지 더 쉽게 이해할 수 있습니다.




## 0. PyTorch 기본 준비

PyTorch에서 행렬은 `torch.tensor()`로 만들 수 있습니다. 행렬의 덧셈, 뺄셈, 스칼라곱, 행렬곱, 전치, 역행렬 계산은 모두 PyTorch 함수로 수행할 수 있습니다.


In [1]:
# PyTorch 라이브러리를 불러옵니다. PyTorch는 행렬과 텐서 계산을 빠르게 처리하는 딥러닝 라이브러리입니다.
import torch  # torch는 행렬, 벡터, 텐서 연산을 수행하기 위해 사용합니다.

# 출력 결과를 보기 좋게 만들기 위해 숫자 표시 형식을 설정합니다.
torch.set_printoptions(precision=4, sci_mode=False)  # 소수점 4자리까지 표시하고 과학적 표기법을 사용하지 않도록 설정합니다.

# 행렬을 이름과 함께 출력하기 위한 보조 함수를 정의합니다.
def print_matrix(name, matrix):  # name은 출력할 행렬 이름이고, matrix는 실제 PyTorch 텐서입니다.
    print(f"\n{name} =")  # 행렬 이름을 먼저 출력하여 어떤 행렬인지 구분합니다.
    print(matrix)  # PyTorch 텐서 형태의 행렬 값을 출력합니다.
    print(f"shape: {tuple(matrix.shape)}")  # 행렬의 크기, 즉 행과 열의 개수를 출력합니다.


## 1.행렬, 행벡터, 열벡터

행렬은 숫자를 직사각형 형태로 배열한 것입니다. 가로 방향을 행(row), 세로 방향을 열(column)이라고 합니다. 행이 하나뿐이면 행벡터, 열이 하나뿐이면 열벡터라고 합니다.


In [2]:
# 2행 3열 행렬 A를 생성합니다. 각 숫자는 행렬의 원소 또는 성분입니다.
A = torch.tensor([[1, -3, 0],  # 첫 번째 행에는 1, -3, 0이 들어갑니다.
                  [7,  5, 9]], # 두 번째 행에는 7, 5, 9가 들어갑니다.
                 dtype=torch.float32)  # 행렬 계산을 위해 실수형 float32로 저장합니다.

# 1행 5열 행벡터를 생성합니다. 행벡터는 행이 1개인 행렬입니다.
row_vector = torch.tensor([[1, 3, -9, 0, 4]], dtype=torch.float32)  # 하나의 행에 여러 값이 들어갑니다.

# 4행 1열 열벡터를 생성합니다. 열벡터는 열이 1개인 행렬입니다.
column_vector = torch.tensor([[2],   # 첫 번째 행의 값입니다.
                              [-5],  # 두 번째 행의 값입니다.
                              [0],   # 세 번째 행의 값입니다.
                              [7]],  # 네 번째 행의 값입니다.
                             dtype=torch.float32)  # 벡터 계산을 위해 실수형으로 저장합니다.

# 생성한 일반 행렬 A를 출력합니다.
print_matrix("A: 2행 3열 행렬", A)  # A의 값과 크기를 확인합니다.

# 생성한 행벡터를 출력합니다.
print_matrix("row_vector: 1행 5열 행벡터", row_vector)  # 행벡터의 모양이 (1, 5)인지 확인합니다.

# 생성한 열벡터를 출력합니다.
print_matrix("column_vector: 4행 1열 열벡터", column_vector)  # 열벡터의 모양이 (4, 1)인지 확인합니다.



A: 2행 3열 행렬 =
tensor([[ 1., -3.,  0.],
        [ 7.,  5.,  9.]])
shape: (2, 3)

row_vector: 1행 5열 행벡터 =
tensor([[ 1.,  3., -9.,  0.,  4.]])
shape: (1, 5)

column_vector: 4행 1열 열벡터 =
tensor([[ 2.],
        [-5.],
        [ 0.],
        [ 7.]])
shape: (4, 1)


## 2. 대각성분, 정사각행렬, 대각행렬

대각성분은 행 번호와 열 번호가 같은 위치의 원소입니다. 정사각행렬은 행과 열의 개수가 같은 행렬입니다. 대각행렬은 정사각행렬 중에서 대각성분을 제외한 나머지 원소가 모두 0인 행렬입니다.


In [3]:
# 3행 3열 정사각행렬 S를 생성합니다. 정사각행렬은 행 수와 열 수가 같습니다.
S = torch.tensor([[1, -3, 0],   # 첫 번째 행입니다.
                  [7,  5, 9],   # 두 번째 행입니다.
                  [2,  1, 8]],  # 세 번째 행입니다.
                 dtype=torch.float32)  # 대각성분 추출과 행렬 연산을 위해 실수형으로 저장합니다.

# torch.diagonal 함수로 대각성분을 추출합니다.
diag_entries = torch.diagonal(S)  # S의 a11, a22, a33 위치에 있는 값을 가져옵니다.

# torch.diag 함수로 대각행렬을 생성합니다.
diagonal_matrix = torch.diag(torch.tensor([6, 3, -2], dtype=torch.float32))  # 주어진 값만 대각성분으로 배치하고 나머지는 0으로 둡니다.

# 정사각행렬 S를 출력합니다.
print_matrix("S: 3행 3열 정사각행렬", S)  # 정사각행렬의 크기와 값을 확인합니다.

# 대각성분을 출력합니다.
print_matrix("S의 대각성분", diag_entries)  # 대각성분이 [1, 5, 8]인지 확인합니다.

# 대각행렬을 출력합니다.
print_matrix("diagonal_matrix: 대각행렬", diagonal_matrix)  # 대각 위치 외에는 모두 0인지 확인합니다.



S: 3행 3열 정사각행렬 =
tensor([[ 1., -3.,  0.],
        [ 7.,  5.,  9.],
        [ 2.,  1.,  8.]])
shape: (3, 3)

S의 대각성분 =
tensor([1., 5., 8.])
shape: (3,)

diagonal_matrix: 대각행렬 =
tensor([[ 6.,  0.,  0.],
        [ 0.,  3.,  0.],
        [ 0.,  0., -2.]])
shape: (3, 3)


## 단위행렬, 행렬의 같음, 덧셈, 스칼라곱, 뺄셈, 영행렬

행렬의 덧셈과 뺄셈은 같은 위치의 원소끼리 계산합니다. 스칼라곱은 행렬의 모든 원소에 같은 숫자를 곱하는 연산입니다. 단위행렬은 행렬 곱셈에서 숫자 1과 같은 역할을 하며, 영행렬은 행렬 덧셈에서 숫자 0과 같은 역할을 합니다.


In [4]:
# 강의자료 예제와 같은 2행 3열 행렬 A를 생성합니다.
A = torch.tensor([[1, -3, 0],  # A의 첫 번째 행입니다.
                  [7,  5, 9]], # A의 두 번째 행입니다.
                 dtype=torch.float32)  # 계산을 위해 실수형으로 저장합니다.

# A와 같은 크기를 갖는 2행 3열 행렬 B를 생성합니다.
B = torch.tensor([[0, 1, 2],   # B의 첫 번째 행입니다.
                  [4, 0, 8]],  # B의 두 번째 행입니다.
                 dtype=torch.float32)  # 계산을 위해 실수형으로 저장합니다.

# A와 같은 값을 가진 행렬 A_copy를 생성합니다.
A_copy = torch.tensor([[1, -3, 0],  # A와 동일한 첫 번째 행입니다.
                       [7,  5, 9]], # A와 동일한 두 번째 행입니다.
                      dtype=torch.float32)  # 비교 연산을 위해 같은 자료형으로 저장합니다.

# torch.equal은 두 텐서의 크기와 모든 원소가 완전히 같은지 확인합니다.
is_equal = torch.equal(A, A_copy)  # A와 A_copy가 같은 행렬인지 True 또는 False로 확인합니다.

# A와 B를 더합니다. 같은 위치에 있는 원소끼리 더해집니다.
A_plus_B = A + B  # 예: 첫 번째 행 두 번째 열은 -3 + 1 = -2가 됩니다.

# 스칼라 c를 정의합니다. 스칼라는 하나의 숫자입니다.
c = 2  # 행렬 전체에 곱할 숫자입니다.

# 스칼라곱을 수행합니다. A의 모든 원소에 2가 곱해집니다.
cA = c * A  # 예: -3은 -6이 되고 7은 14가 됩니다.

# 행렬 뺄셈을 수행합니다. A - B는 A + (-1)B와 같습니다.
A_minus_B = A - B  # 같은 위치의 원소끼리 빼기를 수행합니다.

# A와 같은 크기의 영행렬을 생성합니다.
O = torch.zeros_like(A)  # A와 동일한 shape를 가지며 모든 원소가 0인 행렬입니다.

# 3행 3열 단위행렬을 생성합니다.
I3 = torch.eye(3, dtype=torch.float32)  # 대각성분은 1이고 나머지는 0인 3x3 단위행렬입니다.

# 비교 결과를 출력합니다.
print("A와 A_copy는 같은 행렬인가?", is_equal)  # True이면 두 행렬이 완전히 같다는 뜻입니다.

# 행렬 A를 출력합니다.
print_matrix("A", A)  # A의 값을 확인합니다.

# 행렬 B를 출력합니다.
print_matrix("B", B)  # B의 값을 확인합니다.

# 행렬 덧셈 결과를 출력합니다.
print_matrix("A + B", A_plus_B)  # 강의자료의 덧셈 결과와 비교합니다.

# 스칼라곱 결과를 출력합니다.
print_matrix("2A", cA)  # A의 모든 원소가 2배가 되었는지 확인합니다.

# 행렬 뺄셈 결과를 출력합니다.
print_matrix("A - B", A_minus_B)  # 강의자료의 뺄셈 결과와 비교합니다.

# 영행렬을 출력합니다.
print_matrix("O: 영행렬", O)  # 모든 원소가 0인지 확인합니다.

# 단위행렬을 출력합니다.
print_matrix("I3: 3차 단위행렬", I3)  # 대각성분이 1이고 나머지가 0인지 확인합니다.


A와 A_copy는 같은 행렬인가? True

A =
tensor([[ 1., -3.,  0.],
        [ 7.,  5.,  9.]])
shape: (2, 3)

B =
tensor([[0., 1., 2.],
        [4., 0., 8.]])
shape: (2, 3)

A + B =
tensor([[ 1., -2.,  2.],
        [11.,  5., 17.]])
shape: (2, 3)

2A =
tensor([[ 2., -6.,  0.],
        [14., 10., 18.]])
shape: (2, 3)

A - B =
tensor([[ 1., -4., -2.],
        [ 3.,  5.,  1.]])
shape: (2, 3)

O: 영행렬 =
tensor([[0., 0., 0.],
        [0., 0., 0.]])
shape: (2, 3)

I3: 3차 단위행렬 =
tensor([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]])
shape: (3, 3)


## 4.행렬 덧셈과 스칼라배의 성질 확인

행렬의 덧셈과 스칼라곱은 일반 숫자 계산과 비슷한 성질을 가집니다. 예를 들어 `A + B = B + A`, `A + O = A`, `c(A+B)=cA+cB` 같은 성질을 PyTorch로 직접 확인할 수 있습니다.


In [5]:
# 성질 확인을 위해 A와 같은 크기의 행렬 C를 생성합니다.
C = torch.tensor([[2, 0, -1],  # C의 첫 번째 행입니다.
                  [3, 1,  4]], # C의 두 번째 행입니다.
                 dtype=torch.float32)  # 덧셈과 스칼라곱을 위해 실수형으로 저장합니다.

# 스칼라 c와 d를 정의합니다.
c = 2  # 첫 번째 스칼라입니다.
d = 3  # 두 번째 스칼라입니다.

# 교환법칙 A + B = B + A를 확인합니다.
commutative = torch.allclose(A + B, B + A)  # 두 결과가 같은지 확인합니다.

# 결합법칙 (A+B)+C = A+(B+C)를 확인합니다.
associative = torch.allclose((A + B) + C, A + (B + C))  # 세 행렬의 덧셈 순서를 바꾸어도 같은지 확인합니다.

# 영행렬 성질 A + O = A를 확인합니다.
zero_property = torch.allclose(A + O, A)  # 영행렬을 더해도 A가 그대로인지 확인합니다.

# 덧셈 역원 성질 A + (-A) = O를 확인합니다.
add_inverse = torch.allclose(A + (-A), O)  # A와 -A를 더하면 영행렬이 되는지 확인합니다.

# 분배법칙 c(A+B) = cA + cB를 확인합니다.
distributive = torch.allclose(c * (A + B), c * A + c * B)  # 스칼라곱이 덧셈에 대해 분배되는지 확인합니다.

# 스칼라곱 결합법칙 c(dA) = (cd)A를 확인합니다.
scalar_associative = torch.allclose(c * (d * A), (c * d) * A)  # 스칼라를 곱하는 순서가 결과에 영향을 주지 않는지 확인합니다.

# 각 성질의 확인 결과를 출력합니다.
print("A + B = B + A 성립:", commutative)  # True이면 교환법칙이 성립합니다.
print("(A+B)+C = A+(B+C) 성립:", associative)  # True이면 결합법칙이 성립합니다.
print("A + O = A 성립:", zero_property)  # True이면 영행렬 성질이 성립합니다.
print("A + (-A) = O 성립:", add_inverse)  # True이면 덧셈 역원 성질이 성립합니다.
print("c(A+B) = cA + cB 성립:", distributive)  # True이면 분배법칙이 성립합니다.
print("c(dA) = (cd)A 성립:", scalar_associative)  # True이면 스칼라곱 결합법칙이 성립합니다.


A + B = B + A 성립: True
(A+B)+C = A+(B+C) 성립: True
A + O = A 성립: True
A + (-A) = O 성립: True
c(A+B) = cA + cB 성립: True
c(dA) = (cd)A 성립: True


## 5.행렬 곱셈, 거듭제곱, 곱셈 성질

행렬 곱셈은 앞 행렬의 행과 뒤 행렬의 열을 곱한 뒤 더하는 방식으로 계산합니다. PyTorch에서는 `@` 연산자 또는 `torch.matmul()`을 사용합니다.


In [6]:
# 행렬 곱셈 예제에서 사용할 2행 3열 행렬 A를 생성합니다.
A = torch.tensor([[1, -3, 0],  # A의 첫 번째 행입니다.
                  [7,  5, 9]], # A의 두 번째 행입니다.
                 dtype=torch.float32)  # 행렬 곱셈을 위해 실수형으로 저장합니다.

# A와 곱할 3행 2열 행렬 B를 생성합니다.
B = torch.tensor([[0, 4],  # B의 첫 번째 행입니다.
                  [1, 0],  # B의 두 번째 행입니다.
                  [2, 8]], # B의 세 번째 행입니다.
                 dtype=torch.float32)  # 행렬 곱셈을 위해 실수형으로 저장합니다.

# @ 연산자는 PyTorch에서 행렬 곱셈을 의미합니다.
AB = A @ B  # A의 shape가 (2, 3), B의 shape가 (3, 2)이므로 결과는 (2, 2)입니다.

# c11을 직접 계산합니다. A의 첫 번째 행과 B의 첫 번째 열을 곱해 더합니다.
c11_manual = A[0, 0] * B[0, 0] + A[0, 1] * B[1, 0] + A[0, 2] * B[2, 0]  # 1*0 + (-3)*1 + 0*2 = -3입니다.

# 행렬 A를 출력합니다.
print_matrix("A", A)  # 행렬 곱셈의 왼쪽 행렬입니다.

# 행렬 B를 출력합니다.
print_matrix("B", B)  # 행렬 곱셈의 오른쪽 행렬입니다.

# 행렬 곱셈 결과를 출력합니다.
print_matrix("AB", AB)  # 강의자료의 결과인 [[-3, 4], [23, 100]]과 같은지 확인합니다.

# 직접 계산한 c11 값을 출력합니다.
print("c11 직접 계산 결과:", c11_manual.item())  # item()은 텐서 값 하나를 파이썬 숫자로 변환합니다.



A =
tensor([[ 1., -3.,  0.],
        [ 7.,  5.,  9.]])
shape: (2, 3)

B =
tensor([[0., 4.],
        [1., 0.],
        [2., 8.]])
shape: (3, 2)

AB =
tensor([[ -3.,   4.],
        [ 23., 100.]])
shape: (2, 2)
c11 직접 계산 결과: -3.0


In [7]:
# 행렬의 거듭제곱 예제에서 사용할 2행 2열 정사각행렬 P를 생성합니다.
P = torch.tensor([[6, 7],  # P의 첫 번째 행입니다.
                  [0, 1]], # P의 두 번째 행입니다.
                 dtype=torch.float32)  # 거듭제곱 계산을 위해 실수형으로 저장합니다.

# torch.linalg.matrix_power 함수로 P의 1제곱을 계산합니다.
P1 = torch.linalg.matrix_power(P, 1)  # 1제곱은 자기 자신과 같습니다.

# torch.linalg.matrix_power 함수로 P의 2제곱을 계산합니다.
P2 = torch.linalg.matrix_power(P, 2)  # 2제곱은 P @ P와 같습니다.

# 직접 P @ P도 계산합니다.
P2_manual = P @ P  # 행렬 곱셈을 이용해 P의 제곱을 직접 계산합니다.

# P를 출력합니다.
print_matrix("P", P)  # 원래 행렬을 확인합니다.

# P의 1제곱을 출력합니다.
print_matrix("P^1", P1)  # P와 같은지 확인합니다.

# P의 2제곱을 출력합니다.
print_matrix("P^2", P2)  # 강의자료의 결과와 비교합니다.

# 직접 계산한 P @ P를 출력합니다.
print_matrix("P @ P", P2_manual)  # matrix_power 결과와 같은지 확인합니다.



P =
tensor([[6., 7.],
        [0., 1.]])
shape: (2, 2)

P^1 =
tensor([[6., 7.],
        [0., 1.]])
shape: (2, 2)

P^2 =
tensor([[36., 49.],
        [ 0.,  1.]])
shape: (2, 2)

P @ P =
tensor([[36., 49.],
        [ 0.,  1.]])
shape: (2, 2)


In [8]:
# 행렬 곱셈 성질 확인을 위해 정사각행렬 M1을 생성합니다.
M1 = torch.tensor([[1, 2],  # M1의 첫 번째 행입니다.
                   [3, 4]], # M1의 두 번째 행입니다.
                  dtype=torch.float32)  # 성질 확인을 위해 실수형으로 저장합니다.

# 행렬 곱셈 성질 확인을 위해 정사각행렬 M2를 생성합니다.
M2 = torch.tensor([[0, 1],  # M2의 첫 번째 행입니다.
                   [2, 3]], # M2의 두 번째 행입니다.
                  dtype=torch.float32)  # 성질 확인을 위해 실수형으로 저장합니다.

# 행렬 곱셈 성질 확인을 위해 정사각행렬 M3를 생성합니다.
M3 = torch.tensor([[5, 1],  # M3의 첫 번째 행입니다.
                   [0, 2]], # M3의 두 번째 행입니다.
                  dtype=torch.float32)  # 성질 확인을 위해 실수형으로 저장합니다.

# 2행 2열 단위행렬을 생성합니다.
I2 = torch.eye(2, dtype=torch.float32)  # 행렬 곱셈에서 숫자 1과 같은 역할을 합니다.

# 결합법칙 M1(M2M3) = (M1M2)M3를 확인합니다.
matmul_associative = torch.allclose(M1 @ (M2 @ M3), (M1 @ M2) @ M3)  # 행렬 곱셈 순서의 묶음이 바뀌어도 같은지 확인합니다.

# 왼쪽 분배법칙 M1(M2+M3) = M1M2 + M1M3를 확인합니다.
left_distributive = torch.allclose(M1 @ (M2 + M3), M1 @ M2 + M1 @ M3)  # 곱셈이 덧셈에 대해 분배되는지 확인합니다.

# 오른쪽 분배법칙 (M1+M2)M3 = M1M3 + M2M3를 확인합니다.
right_distributive = torch.allclose((M1 + M2) @ M3, M1 @ M3 + M2 @ M3)  # 오른쪽 곱셈에서도 분배법칙이 성립하는지 확인합니다.

# 단위행렬 성질 I2M1 = M1 = M1I2를 확인합니다.
identity_property = torch.allclose(I2 @ M1, M1) and torch.allclose(M1 @ I2, M1)  # 단위행렬을 곱해도 원래 행렬이 유지되는지 확인합니다.

# 각 성질의 결과를 출력합니다.
print("M1(M2M3) = (M1M2)M3 성립:", matmul_associative)  # True이면 행렬 곱셈 결합법칙이 성립합니다.
print("M1(M2+M3) = M1M2 + M1M3 성립:", left_distributive)  # True이면 왼쪽 분배법칙이 성립합니다.
print("(M1+M2)M3 = M1M3 + M2M3 성립:", right_distributive)  # True이면 오른쪽 분배법칙이 성립합니다.
print("I2M1 = M1 = M1I2 성립:", identity_property)  # True이면 단위행렬 성질이 성립합니다.


M1(M2M3) = (M1M2)M3 성립: True
M1(M2+M3) = M1M2 + M1M3 성립: True
(M1+M2)M3 = M1M3 + M2M3 성립: True
I2M1 = M1 = M1I2 성립: True


## 6. 전치행렬과 전치행렬의 성질

전치행렬은 행과 열을 서로 바꾼 행렬입니다. PyTorch에서는 `.T` 또는 `.transpose()`를 사용할 수 있습니다. 벡터의 내적은 `u.T @ v` 형태로 표현할 수 있습니다.


In [9]:
# 전치 예제에 사용할 2행 3열 행렬 A를 생성합니다.
A = torch.tensor([[1, -3, 0],  # A의 첫 번째 행입니다.
                  [7,  5, 9]], # A의 두 번째 행입니다.
                 dtype=torch.float32)  # 전치 연산을 위해 실수형으로 저장합니다.

# A.T는 A의 전치행렬입니다.
A_T = A.T  # 2행 3열 행렬이 3행 2열 행렬로 바뀝니다.

# 열벡터 u를 생성합니다.
u = torch.tensor([[1],  # u의 첫 번째 성분입니다.
                  [2],  # u의 두 번째 성분입니다.
                  [3]], # u의 세 번째 성분입니다.
                 dtype=torch.float32)  # 내적 계산을 위해 실수형으로 저장합니다.

# 열벡터 v를 생성합니다.
v = torch.tensor([[4],  # v의 첫 번째 성분입니다.
                  [5],  # v의 두 번째 성분입니다.
                  [6]], # v의 세 번째 성분입니다.
                 dtype=torch.float32)  # 내적 계산을 위해 실수형으로 저장합니다.

# u.T @ v는 두 열벡터의 내적입니다.
dot_product = u.T @ v  # 1*4 + 2*5 + 3*6 = 32입니다.

# 원래 행렬을 출력합니다.
print_matrix("A", A)  # 전치하기 전 행렬입니다.

# 전치행렬을 출력합니다.
print_matrix("A.T", A_T)  # 행과 열이 바뀐 결과입니다.

# 벡터 u를 출력합니다.
print_matrix("u", u)  # 내적의 첫 번째 벡터입니다.

# 벡터 v를 출력합니다.
print_matrix("v", v)  # 내적의 두 번째 벡터입니다.

# 내적 결과를 출력합니다.
print_matrix("u.T @ v", dot_product)  # 결과가 1행 1열 텐서로 출력됩니다.



A =
tensor([[ 1., -3.,  0.],
        [ 7.,  5.,  9.]])
shape: (2, 3)

A.T =
tensor([[ 1.,  7.],
        [-3.,  5.],
        [ 0.,  9.]])
shape: (3, 2)

u =
tensor([[1.],
        [2.],
        [3.]])
shape: (3, 1)

v =
tensor([[4.],
        [5.],
        [6.]])
shape: (3, 1)

u.T @ v =
tensor([[32.]])
shape: (1, 1)


In [10]:
# 전치행렬 성질 확인을 위해 2행 2열 행렬 A를 생성합니다.
A = torch.tensor([[1, 2],  # A의 첫 번째 행입니다.
                  [3, 4]], # A의 두 번째 행입니다.
                 dtype=torch.float32)  # 전치 성질 확인을 위해 실수형으로 저장합니다.

# 전치행렬 성질 확인을 위해 2행 2열 행렬 B를 생성합니다.
B = torch.tensor([[0, 1],  # B의 첫 번째 행입니다.
                  [2, 3]], # B의 두 번째 행입니다.
                 dtype=torch.float32)  # 전치 성질 확인을 위해 실수형으로 저장합니다.

# 스칼라 k를 정의합니다.
k = 5  # 스칼라곱의 전치 성질을 확인하기 위한 숫자입니다.

# (A.T).T = A 성질을 확인합니다.
transpose_twice = torch.allclose(A.T.T, A)  # 두 번 전치하면 원래 행렬이 되는지 확인합니다.

# (A+B).T = A.T + B.T 성질을 확인합니다.
transpose_sum = torch.allclose((A + B).T, A.T + B.T)  # 덧셈 후 전치와 전치 후 덧셈이 같은지 확인합니다.

# (kA).T = kA.T 성질을 확인합니다.
transpose_scalar = torch.allclose((k * A).T, k * A.T)  # 스칼라곱과 전치의 순서를 바꾸어도 같은지 확인합니다.

# (AB).T = B.T A.T 성질을 확인합니다.
transpose_product = torch.allclose((A @ B).T, B.T @ A.T)  # 곱의 전치는 순서가 바뀌는지 확인합니다.

# 전치 성질 확인 결과를 출력합니다.
print("(A.T).T = A 성립:", transpose_twice)  # True이면 두 번 전치 성질이 성립합니다.
print("(A+B).T = A.T + B.T 성립:", transpose_sum)  # True이면 합의 전치 성질이 성립합니다.
print("(kA).T = kA.T 성립:", transpose_scalar)  # True이면 스칼라곱 전치 성질이 성립합니다.
print("(AB).T = B.T A.T 성립:", transpose_product)  # True이면 곱의 전치 성질이 성립합니다.


(A.T).T = A 성립: True
(A+B).T = A.T + B.T 성립: True
(kA).T = kA.T 성립: True
(AB).T = B.T A.T 성립: True


## 7. 분할행렬과 여러 가지 행렬 곱 표현

분할행렬은 큰 행렬을 여러 개의 작은 블록으로 나누어 표현하는 방법입니다. 행렬 곱셈은 열 단위, 행 단위, 행-열 단위, 열-행 단위로 다양하게 해석할 수 있습니다.


In [11]:
# 분할행렬 예제에 사용할 3행 3열 행렬 X를 생성합니다.
X = torch.tensor([[ 1,  2,   0],  # X의 첫 번째 행입니다.
                  [ 3,  4,   0],  # X의 두 번째 행입니다.
                  [-1, -2, 100]], # X의 세 번째 행입니다.
                 dtype=torch.float32)  # 블록 분할과 연산을 위해 실수형으로 저장합니다.

# X의 왼쪽 위 블록 A_block을 추출합니다.
A_block = X[:2, :2]  # 앞의 2행과 앞의 2열을 가져옵니다.

# X의 오른쪽 위 블록 B_block을 추출합니다.
B_block = X[:2, 2:]  # 앞의 2행과 마지막 1열을 가져옵니다.

# X의 왼쪽 아래 블록 C_block을 추출합니다.
C_block = X[2:, :2]  # 마지막 1행과 앞의 2열을 가져옵니다.

# X의 오른쪽 아래 블록 D_block을 추출합니다.
D_block = X[2:, 2:]  # 마지막 1행과 마지막 1열을 가져옵니다.

# 원래 행렬 X를 출력합니다.
print_matrix("X", X)  # 분할하기 전 전체 행렬을 확인합니다.

# 각 블록을 출력합니다.
print_matrix("A_block: 왼쪽 위 블록", A_block)  # 왼쪽 위 2x2 블록입니다.
print_matrix("B_block: 오른쪽 위 블록", B_block)  # 오른쪽 위 2x1 블록입니다.
print_matrix("C_block: 왼쪽 아래 블록", C_block)  # 왼쪽 아래 1x2 블록입니다.
print_matrix("D_block: 오른쪽 아래 블록", D_block)  # 오른쪽 아래 1x1 블록입니다.



X =
tensor([[  1.,   2.,   0.],
        [  3.,   4.,   0.],
        [ -1.,  -2., 100.]])
shape: (3, 3)

A_block: 왼쪽 위 블록 =
tensor([[1., 2.],
        [3., 4.]])
shape: (2, 2)

B_block: 오른쪽 위 블록 =
tensor([[0.],
        [0.]])
shape: (2, 1)

C_block: 왼쪽 아래 블록 =
tensor([[-1., -2.]])
shape: (1, 2)

D_block: 오른쪽 아래 블록 =
tensor([[100.]])
shape: (1, 1)


In [12]:
# 행렬-열 표현을 확인하기 위해 A를 생성합니다.
A = torch.tensor([[1, -3, 0],  # A의 첫 번째 행입니다.
                  [7,  5, 9]], # A의 두 번째 행입니다.
                 dtype=torch.float32)  # 행렬 곱셈을 위해 실수형으로 저장합니다.

# 행렬-열 표현을 확인하기 위해 B를 생성합니다.
B = torch.tensor([[0, 4],  # B의 첫 번째 행입니다.
                  [1, 0],  # B의 두 번째 행입니다.
                  [2, 8]], # B의 세 번째 행입니다.
                 dtype=torch.float32)  # 행렬 곱셈을 위해 실수형으로 저장합니다.

# B의 첫 번째 열 b1을 추출합니다.
b1 = B[:, 0:1]  # 모든 행에서 첫 번째 열만 가져와 열벡터 형태를 유지합니다.

# B의 두 번째 열 b2를 추출합니다.
b2 = B[:, 1:2]  # 모든 행에서 두 번째 열만 가져와 열벡터 형태를 유지합니다.

# A와 b1을 곱합니다.
Ab1 = A @ b1  # 결과는 AB의 첫 번째 열이 됩니다.

# A와 b2를 곱합니다.
Ab2 = A @ b2  # 결과는 AB의 두 번째 열이 됩니다.

# 두 결과 열을 옆으로 붙입니다.
AB_by_columns = torch.cat([Ab1, Ab2], dim=1)  # 열 방향으로 결합하여 AB를 구성합니다.

# 일반적인 행렬 곱셈 결과를 계산합니다.
AB = A @ B  # 비교를 위한 일반 행렬 곱셈 결과입니다.

# 행렬-열 표현 결과를 출력합니다.
print_matrix("A @ b1", Ab1)  # AB의 첫 번째 열입니다.
print_matrix("A @ b2", Ab2)  # AB의 두 번째 열입니다.
print_matrix("열 단위로 만든 AB", AB_by_columns)  # 열 단위 계산을 합친 결과입니다.
print_matrix("일반 행렬 곱 AB", AB)  # 일반 행렬곱 결과와 비교합니다.



A @ b1 =
tensor([[-3.],
        [23.]])
shape: (2, 1)

A @ b2 =
tensor([[  4.],
        [100.]])
shape: (2, 1)

열 단위로 만든 AB =
tensor([[ -3.,   4.],
        [ 23., 100.]])
shape: (2, 2)

일반 행렬 곱 AB =
tensor([[ -3.,   4.],
        [ 23., 100.]])
shape: (2, 2)


In [13]:
# 행-행렬 표현을 위해 A의 첫 번째 행을 추출합니다.
A1 = A[0:1, :]  # 첫 번째 행만 가져오되 2차원 행렬 형태를 유지합니다.

# 행-행렬 표현을 위해 A의 두 번째 행을 추출합니다.
A2 = A[1:2, :]  # 두 번째 행만 가져오되 2차원 행렬 형태를 유지합니다.

# A1과 B를 곱합니다.
A1B = A1 @ B  # 결과는 AB의 첫 번째 행이 됩니다.

# A2와 B를 곱합니다.
A2B = A2 @ B  # 결과는 AB의 두 번째 행이 됩니다.

# 두 결과 행을 아래로 붙입니다.
AB_by_rows = torch.cat([A1B, A2B], dim=0)  # 행 방향으로 결합하여 AB를 구성합니다.

# 행 단위 표현 결과를 출력합니다.
print_matrix("A1 @ B", A1B)  # AB의 첫 번째 행입니다.
print_matrix("A2 @ B", A2B)  # AB의 두 번째 행입니다.
print_matrix("행 단위로 만든 AB", AB_by_rows)  # 행 단위 계산을 합친 결과입니다.
print_matrix("일반 행렬 곱 AB", AB)  # 일반 행렬곱 결과와 비교합니다.



A1 @ B =
tensor([[-3.,  4.]])
shape: (1, 2)

A2 @ B =
tensor([[ 23., 100.]])
shape: (1, 2)

행 단위로 만든 AB =
tensor([[ -3.,   4.],
        [ 23., 100.]])
shape: (2, 2)

일반 행렬 곱 AB =
tensor([[ -3.,   4.],
        [ 23., 100.]])
shape: (2, 2)


In [14]:
# 행-열 표현은 각 원소를 행벡터와 열벡터의 내적으로 계산합니다.
c11 = A[0:1, :] @ B[:, 0:1]  # AB의 1행 1열 원소를 계산합니다.
c12 = A[0:1, :] @ B[:, 1:2]  # AB의 1행 2열 원소를 계산합니다.
c21 = A[1:2, :] @ B[:, 0:1]  # AB의 2행 1열 원소를 계산합니다.
c22 = A[1:2, :] @ B[:, 1:2]  # AB의 2행 2열 원소를 계산합니다.

# 계산한 네 개의 값을 하나의 2행 2열 행렬로 구성합니다.
AB_by_row_column = torch.tensor([[c11.item(), c12.item()],  # 첫 번째 행에는 c11과 c12를 넣습니다.
                                 [c21.item(), c22.item()]], # 두 번째 행에는 c21과 c22를 넣습니다.
                                dtype=torch.float32)  # 결과 행렬을 실수형으로 생성합니다.

# 행-열 표현 결과를 출력합니다.
print_matrix("행-열 내적으로 만든 AB", AB_by_row_column)  # 각 원소를 직접 계산한 결과입니다.
print_matrix("일반 행렬 곱 AB", AB)  # 일반 행렬곱 결과와 비교합니다.



행-열 내적으로 만든 AB =
tensor([[ -3.,   4.],
        [ 23., 100.]])
shape: (2, 2)

일반 행렬 곱 AB =
tensor([[ -3.,   4.],
        [ 23., 100.]])
shape: (2, 2)


In [15]:
# 열-행 표현을 위해 A의 첫 번째 열을 추출합니다.
a1 = A[:, 0:1]  # A의 첫 번째 열을 열벡터 형태로 가져옵니다.

# 열-행 표현을 위해 A의 두 번째 열을 추출합니다.
a2 = A[:, 1:2]  # A의 두 번째 열을 열벡터 형태로 가져옵니다.

# 열-행 표현을 위해 A의 세 번째 열을 추출합니다.
a3 = A[:, 2:3]  # A의 세 번째 열을 열벡터 형태로 가져옵니다.

# B의 첫 번째 행을 추출합니다.
B1 = B[0:1, :]  # B의 첫 번째 행을 행벡터 형태로 가져옵니다.

# B의 두 번째 행을 추출합니다.
B2 = B[1:2, :]  # B의 두 번째 행을 행벡터 형태로 가져옵니다.

# B의 세 번째 행을 추출합니다.
B3 = B[2:3, :]  # B의 세 번째 행을 행벡터 형태로 가져옵니다.

# 첫 번째 외적을 계산합니다.
outer1 = a1 @ B1  # A의 첫 번째 열과 B의 첫 번째 행을 곱해 2x2 행렬을 만듭니다.

# 두 번째 외적을 계산합니다.
outer2 = a2 @ B2  # A의 두 번째 열과 B의 두 번째 행을 곱해 2x2 행렬을 만듭니다.

# 세 번째 외적을 계산합니다.
outer3 = a3 @ B3  # A의 세 번째 열과 B의 세 번째 행을 곱해 2x2 행렬을 만듭니다.

# 세 외적을 더해 AB를 구성합니다.
AB_by_outer = outer1 + outer2 + outer3  # 외적전개 방식으로 행렬곱 결과를 얻습니다.

# 외적 결과를 각각 출력합니다.
print_matrix("a1 @ B1", outer1)  # 첫 번째 외적 결과입니다.
print_matrix("a2 @ B2", outer2)  # 두 번째 외적 결과입니다.
print_matrix("a3 @ B3", outer3)  # 세 번째 외적 결과입니다.
print_matrix("외적전개로 만든 AB", AB_by_outer)  # 세 외적의 합입니다.
print_matrix("일반 행렬 곱 AB", AB)  # 일반 행렬곱 결과와 비교합니다.



a1 @ B1 =
tensor([[ 0.,  4.],
        [ 0., 28.]])
shape: (2, 2)

a2 @ B2 =
tensor([[-3., -0.],
        [ 5.,  0.]])
shape: (2, 2)

a3 @ B3 =
tensor([[ 0.,  0.],
        [18., 72.]])
shape: (2, 2)

외적전개로 만든 AB =
tensor([[ -3.,   4.],
        [ 23., 100.]])
shape: (2, 2)

일반 행렬 곱 AB =
tensor([[ -3.,   4.],
        [ 23., 100.]])
shape: (2, 2)


In [16]:
# 블록 연산을 위한 행렬 Y를 생성합니다.
Y = torch.tensor([[ 0, 2, -1],  # Y의 첫 번째 행입니다.
                  [ 1, 2,  1],  # Y의 두 번째 행입니다.
                  [-1, 0,  4]], # Y의 세 번째 행입니다.
                 dtype=torch.float32)  # 블록 연산을 위해 실수형으로 저장합니다.

# Y의 왼쪽 위 블록 E를 추출합니다.
E = Y[:2, :2]  # Y의 앞 2행과 앞 2열입니다.

# Y의 오른쪽 위 블록 F를 추출합니다.
F = Y[:2, 2:]  # Y의 앞 2행과 마지막 1열입니다.

# Y의 왼쪽 아래 블록 G를 추출합니다.
G = Y[2:, :2]  # Y의 마지막 1행과 앞 2열입니다.

# Y의 오른쪽 아래 블록 H를 추출합니다.
H = Y[2:, 2:]  # Y의 마지막 1행과 마지막 1열입니다.

# 블록 덧셈을 수행합니다.
block_sum = torch.cat([torch.cat([A_block + E, B_block + F], dim=1),  # 위쪽 블록 행을 만듭니다.
                       torch.cat([C_block + G, D_block + H], dim=1)], # 아래쪽 블록 행을 만듭니다.
                      dim=0)  # 위쪽과 아래쪽 블록 행을 세로로 결합합니다.

# 일반 행렬 덧셈 결과를 계산합니다.
normal_sum = X + Y  # 블록이 아닌 전체 행렬 기준의 덧셈입니다.

# 블록 곱셈을 수행합니다.
block_product = torch.cat([torch.cat([A_block @ E + B_block @ G, A_block @ F + B_block @ H], dim=1),  # 위쪽 블록 행입니다.
                           torch.cat([C_block @ E + D_block @ G, C_block @ F + D_block @ H], dim=1)], # 아래쪽 블록 행입니다.
                          dim=0)  # 블록들을 세로로 결합합니다.

# 일반 행렬 곱셈 결과를 계산합니다.
normal_product = X @ Y  # 전체 행렬을 직접 곱한 결과입니다.

# 블록 덧셈 결과를 출력합니다.
print_matrix("블록 덧셈 결과", block_sum)  # 블록 단위로 계산한 덧셈 결과입니다.

# 일반 덧셈 결과를 출력합니다.
print_matrix("일반 덧셈 결과", normal_sum)  # 전체 행렬 기준의 덧셈 결과입니다.

# 블록 곱셈 결과를 출력합니다.
print_matrix("블록 곱셈 결과", block_product)  # 블록 단위로 계산한 곱셈 결과입니다.

# 일반 곱셈 결과를 출력합니다.
print_matrix("일반 곱셈 결과", normal_product)  # 전체 행렬 기준의 곱셈 결과입니다.



블록 덧셈 결과 =
tensor([[  1.,   4.,  -1.],
        [  4.,   6.,   1.],
        [ -2.,  -2., 104.]])
shape: (3, 3)

일반 덧셈 결과 =
tensor([[  1.,   4.,  -1.],
        [  4.,   6.,   1.],
        [ -2.,  -2., 104.]])
shape: (3, 3)

블록 곱셈 결과 =
tensor([[   2.,    6.,    1.],
        [   4.,   14.,    1.],
        [-102.,   -6.,  399.]])
shape: (3, 3)

일반 곱셈 결과 =
tensor([[   2.,    6.,    1.],
        [   4.,   14.,    1.],
        [-102.,   -6.,  399.]])
shape: (3, 3)


## 8. 선형연립방정식과 행렬방정식 `AX = B`

선형연립방정식은 계수행렬 `A`, 미지수 벡터 `X`, 결과 벡터 `B`를 이용하여 `AX = B` 형태로 표현할 수 있습니다. 이 표현은 머신러닝의 선형회귀, 최적화, 딥러닝의 선형층 계산과 연결됩니다.


In [17]:
# 연립방정식의 계수행렬 A를 생성합니다.
A = torch.tensor([[1,  1, -1],  # 첫 번째 방정식 x + y - z = 1의 계수입니다.
                  [2,  1,  1],  # 두 번째 방정식 2x + y + z = 0의 계수입니다.
                  [1, -3,  0]], # 세 번째 방정식 x - 3y + 0z = 4의 계수입니다.
                 dtype=torch.float32)  # 연립방정식 풀이를 위해 실수형으로 저장합니다.

# 결과 벡터 B를 생성합니다.
B = torch.tensor([[1],  # 첫 번째 방정식의 오른쪽 값입니다.
                  [0],  # 두 번째 방정식의 오른쪽 값입니다.
                  [4]], # 세 번째 방정식의 오른쪽 값입니다.
                 dtype=torch.float32)  # 연립방정식 풀이를 위해 실수형으로 저장합니다.

# torch.linalg.solve 함수로 AX = B를 만족하는 X를 구합니다.
X_solution = torch.linalg.solve(A, B)  # 역행렬을 직접 구하지 않고 안정적으로 해를 계산합니다.

# 계산된 해를 다시 A에 곱해 B가 나오는지 확인합니다.
B_check = A @ X_solution  # AX_solution을 계산하여 원래 B와 같은지 확인합니다.

# 계수행렬 A를 출력합니다.
print_matrix("A: 계수행렬", A)  # 각 방정식의 계수를 행렬로 표현한 것입니다.

# 결과 벡터 B를 출력합니다.
print_matrix("B: 결과 벡터", B)  # 각 방정식의 오른쪽 값을 벡터로 표현한 것입니다.

# 미지수 해 X를 출력합니다.
print_matrix("X: 해 벡터 [x, y, z]", X_solution)  # x, y, z 값을 확인합니다.

# AX가 B와 같은지 확인한 결과를 출력합니다.
print_matrix("A @ X", B_check)  # 계산한 해가 원래 방정식을 만족하는지 확인합니다.



A: 계수행렬 =
tensor([[ 1.,  1., -1.],
        [ 2.,  1.,  1.],
        [ 1., -3.,  0.]])
shape: (3, 3)

B: 결과 벡터 =
tensor([[1.],
        [0.],
        [4.]])
shape: (3, 1)

X: 해 벡터 [x, y, z] =
tensor([[ 1.],
        [-1.],
        [-1.]])
shape: (3, 1)

A @ X =
tensor([[1.],
        [0.],
        [4.]])
shape: (3, 1)


## 9. 역행렬과 역행렬을 이용한 선형연립방정식 풀이

역행렬은 행렬 곱셈에서 나눗셈과 비슷한 역할을 합니다. 정사각행렬 `A`에 대해 `AB = BA = I`가 되는 행렬 `B`를 `A`의 역행렬이라고 하고, `A⁻¹`로 표기합니다. `AX = B`에서 `A⁻¹`이 존재하면 `X = A⁻¹B`로 해를 구할 수 있습니다.


In [18]:
# 역행렬 예제의 행렬 A를 생성합니다.
A = torch.tensor([[1, 2],  # A의 첫 번째 행입니다.
                  [1, 1]], # A의 두 번째 행입니다.
                 dtype=torch.float32)  # 역행렬 계산을 위해 실수형으로 저장합니다.

# 강의자료에서 제시된 A의 역행렬 B를 생성합니다.
B = torch.tensor([[-1,  2],  # B의 첫 번째 행입니다.
                  [ 1, -1]], # B의 두 번째 행입니다.
                 dtype=torch.float32)  # 행렬 곱셈 확인을 위해 실수형으로 저장합니다.

# 2행 2열 단위행렬 I를 생성합니다.
I = torch.eye(2, dtype=torch.float32)  # AB와 BA가 이 행렬과 같으면 B는 A의 역행렬입니다.

# AB를 계산합니다.
AB = A @ B  # A와 B를 곱해 단위행렬이 나오는지 확인합니다.

# BA를 계산합니다.
BA = B @ A  # B와 A를 반대 순서로 곱해도 단위행렬이 나오는지 확인합니다.

# PyTorch로 A의 역행렬을 직접 계산합니다.
A_inv = torch.linalg.inv(A)  # A의 역행렬을 계산합니다.

# 결과를 출력합니다.
print_matrix("A", A)  # 원래 행렬입니다.
print_matrix("B", B)  # 강의자료에서 제시한 역행렬입니다.
print_matrix("A @ B", AB)  # 단위행렬이 나오는지 확인합니다.
print_matrix("B @ A", BA)  # 단위행렬이 나오는지 확인합니다.
print_matrix("torch.linalg.inv(A)", A_inv)  # PyTorch가 계산한 역행렬입니다.
print("AB가 I와 같은가?", torch.allclose(AB, I))  # True이면 AB = I입니다.
print("BA가 I와 같은가?", torch.allclose(BA, I))  # True이면 BA = I입니다.



A =
tensor([[1., 2.],
        [1., 1.]])
shape: (2, 2)

B =
tensor([[-1.,  2.],
        [ 1., -1.]])
shape: (2, 2)

A @ B =
tensor([[1., 0.],
        [0., 1.]])
shape: (2, 2)

B @ A =
tensor([[1., 0.],
        [0., 1.]])
shape: (2, 2)

torch.linalg.inv(A) =
tensor([[-1.,  2.],
        [ 1., -1.]])
shape: (2, 2)
AB가 I와 같은가? True
BA가 I와 같은가? True


In [19]:
# 2x2 행렬의 역행렬 공식 예제 행렬 A를 생성합니다.
A = torch.tensor([[1, 2],  # a=1, b=2입니다.
                  [3, 5]], # c=3, d=5입니다.
                 dtype=torch.float32)  # 행렬식과 역행렬 계산을 위해 실수형으로 저장합니다.

# 2x2 행렬의 각 원소를 변수에 저장합니다.
a = A[0, 0]  # A의 1행 1열 원소입니다.
b = A[0, 1]  # A의 1행 2열 원소입니다.
c = A[1, 0]  # A의 2행 1열 원소입니다.
d = A[1, 1]  # A의 2행 2열 원소입니다.

# 2x2 행렬의 행렬식 det(A) = ad - bc를 계산합니다.
det_A = a * d - b * c  # det(A)가 0이 아니면 역행렬이 존재합니다.

# 2x2 역행렬 공식을 직접 적용합니다.
A_inv_formula = (1 / det_A) * torch.tensor([[d, -b],  # 공식의 첫 번째 행은 [d, -b]입니다.
                                            [-c, a]], # 공식의 두 번째 행은 [-c, a]입니다.
                                           dtype=torch.float32)  # 공식 계산 결과를 실수형 행렬로 만듭니다.

# PyTorch 내장 함수로 역행렬을 계산합니다.
A_inv_torch = torch.linalg.inv(A)  # 공식 결과와 비교하기 위한 역행렬입니다.

# 결과를 출력합니다.
print_matrix("A", A)  # 원래 행렬입니다.
print("det(A) =", det_A.item())  # 행렬식 값을 출력합니다.
print_matrix("공식으로 구한 A^-1", A_inv_formula)  # 직접 공식으로 계산한 역행렬입니다.
print_matrix("PyTorch로 구한 A^-1", A_inv_torch)  # PyTorch 함수로 계산한 역행렬입니다.
print("두 역행렬 결과가 같은가?", torch.allclose(A_inv_formula, A_inv_torch))  # 공식 결과와 PyTorch 결과가 같은지 확인합니다.



A =
tensor([[1., 2.],
        [3., 5.]])
shape: (2, 2)
det(A) = -1.0

공식으로 구한 A^-1 =
tensor([[-5.,  2.],
        [ 3., -1.]])
shape: (2, 2)

PyTorch로 구한 A^-1 =
tensor([[-5.0000,  2.0000],
        [ 3.0000, -1.0000]])
shape: (2, 2)
두 역행렬 결과가 같은가? True


In [20]:
# 역행렬을 이용해 풀 선형연립방정식의 계수행렬 A를 생성합니다.
A = torch.tensor([[1, 2],  # 첫 번째 방정식 x + 2y = 1의 계수입니다.
                  [3, 5]], # 두 번째 방정식 3x + 5y = 4의 계수입니다.
                 dtype=torch.float32)  # 연립방정식 풀이를 위해 실수형으로 저장합니다.

# 오른쪽 결과 벡터 B를 생성합니다.
B = torch.tensor([[1],  # 첫 번째 방정식의 오른쪽 값입니다.
                  [4]], # 두 번째 방정식의 오른쪽 값입니다.
                 dtype=torch.float32)  # 연립방정식 풀이를 위해 실수형으로 저장합니다.

# A의 역행렬을 계산합니다.
A_inv = torch.linalg.inv(A)  # AX = B에서 X = A^-1B를 계산하기 위해 사용합니다.

# 역행렬을 이용해 해 X를 계산합니다.
X_by_inverse = A_inv @ B  # X = A^-1B입니다.

# torch.linalg.solve로도 해를 계산합니다.
X_by_solve = torch.linalg.solve(A, B)  # 실제 코드에서는 inv보다 solve가 더 안정적입니다.

# 계산한 해가 원래 방정식을 만족하는지 확인합니다.
B_check = A @ X_by_inverse  # AX를 다시 계산하여 B와 같은지 확인합니다.

# 결과를 출력합니다.
print_matrix("A", A)  # 계수행렬입니다.
print_matrix("B", B)  # 결과 벡터입니다.
print_matrix("A^-1", A_inv)  # 역행렬입니다.
print_matrix("X = A^-1B", X_by_inverse)  # 역행렬로 구한 해입니다.
print_matrix("torch.linalg.solve로 구한 X", X_by_solve)  # solve 함수로 구한 해입니다.
print_matrix("A @ X", B_check)  # 해가 원래 방정식을 만족하는지 확인합니다.



A =
tensor([[1., 2.],
        [3., 5.]])
shape: (2, 2)

B =
tensor([[1.],
        [4.]])
shape: (2, 1)

A^-1 =
tensor([[-5.0000,  2.0000],
        [ 3.0000, -1.0000]])
shape: (2, 2)

X = A^-1B =
tensor([[ 3.0000],
        [-1.0000]])
shape: (2, 1)

torch.linalg.solve로 구한 X =
tensor([[ 3.0000],
        [-1.0000]])
shape: (2, 1)

A @ X =
tensor([[1.0000],
        [4.0000]])
shape: (2, 1)


## 10. 가역행렬의 성질 확인

가역행렬은 역행렬이 존재하는 행렬입니다. 가역행렬의 역행렬, 스칼라배, 곱, 전치, 거듭제곱도 조건을 만족하면 다시 가역행렬이 됩니다.


In [21]:
# 가역행렬 성질 확인을 위해 행렬 A를 생성합니다.
A = torch.tensor([[1, 2],  # A의 첫 번째 행입니다.
                  [3, 5]], # A의 두 번째 행입니다.
                 dtype=torch.float32)  # 역행렬 계산을 위해 실수형으로 저장합니다.

# 또 다른 가역행렬 B를 생성합니다.
B = torch.tensor([[2, 1],  # B의 첫 번째 행입니다.
                  [1, 1]], # B의 두 번째 행입니다.
                 dtype=torch.float32)  # 역행렬 계산을 위해 실수형으로 저장합니다.

# 0이 아닌 스칼라 c를 정의합니다.
c = 3.0  # cA의 역행렬 성질을 확인하기 위한 스칼라입니다.

# A의 역행렬을 계산합니다.
A_inv = torch.linalg.inv(A)  # A^-1입니다.

# (A^-1)^-1 = A 성질을 확인합니다.
property_1 = torch.allclose(torch.linalg.inv(A_inv), A)  # 역행렬의 역행렬이 원래 행렬인지 확인합니다.

# (cA)^-1 = (1/c)A^-1 성질을 확인합니다.
property_2 = torch.allclose(torch.linalg.inv(c * A), (1 / c) * A_inv)  # 스칼라배의 역행렬 성질을 확인합니다.

# (AB)^-1 = B^-1 A^-1 성질을 확인합니다.
property_3 = torch.allclose(torch.linalg.inv(A @ B), torch.linalg.inv(B) @ torch.linalg.inv(A))  # 곱의 역행렬은 순서가 바뀌는지 확인합니다.

# (A.T)^-1 = (A^-1).T 성질을 확인합니다.
property_4 = torch.allclose(torch.linalg.inv(A.T), A_inv.T)  # 전치행렬의 역행렬 성질을 확인합니다.

# (A^2)^-1 = (A^-1)^2 성질을 확인합니다.
property_5 = torch.allclose(torch.linalg.inv(torch.linalg.matrix_power(A, 2)), torch.linalg.matrix_power(A_inv, 2))  # 거듭제곱의 역행렬 성질을 확인합니다.

# 결과를 출력합니다.
print("(A^-1)^-1 = A 성립:", property_1)  # True이면 성질이 성립합니다.
print("(cA)^-1 = (1/c)A^-1 성립:", property_2)  # True이면 성질이 성립합니다.
print("(AB)^-1 = B^-1A^-1 성립:", property_3)  # True이면 성질이 성립합니다.
print("(A.T)^-1 = (A^-1).T 성립:", property_4)  # True이면 성질이 성립합니다.
print("(A^2)^-1 = (A^-1)^2 성립:", property_5)  # True이면 성질이 성립합니다.


(A^-1)^-1 = A 성립: True
(cA)^-1 = (1/c)A^-1 성립: True
(AB)^-1 = B^-1A^-1 성립: True
(A.T)^-1 = (A^-1).T 성립: True
(A^2)^-1 = (A^-1)^2 성립: True


## 11.영공간(Null Space)

영공간은 `AX = 0`을 만족하는 모든 벡터 `X`의 집합입니다. 강의자료의 예제 `A = [[1, 2], [-2, -4]]`에서는 `x + 2y = 0`이므로 `x = -2t`, `y = t`가 됩니다. 따라서 영공간의 벡터는 `[-2t, t]` 형태입니다.


In [22]:
# 영공간 예제의 행렬 A를 생성합니다.
A = torch.tensor([[ 1,  2],  # 첫 번째 행은 x + 2y = 0을 의미합니다.
                  [-2, -4]], # 두 번째 행은 -2x - 4y = 0으로 첫 번째 식의 -2배입니다.
                 dtype=torch.float32)  # 영공간 계산 확인을 위해 실수형으로 저장합니다.

# 매개변수 t 값을 하나 정합니다.
t = torch.tensor(3.0)  # t는 임의의 실수이며 여기서는 예시로 3을 사용합니다.

# 강의자료의 영공간 형태 X = [-2t, t]를 만듭니다.
X_null = torch.tensor([[-2 * t],  # x = -2t입니다.
                       [ t    ]], # y = t입니다.
                      dtype=torch.float32)  # 행렬 곱셈을 위해 열벡터 형태로 저장합니다.

# A @ X_null을 계산합니다.
AX = A @ X_null  # 결과가 0벡터이면 X_null은 A의 영공간에 속합니다.

# PyTorch의 SVD를 이용해 영공간 방향도 확인합니다.
U, S, Vh = torch.linalg.svd(A)  # SVD는 행렬을 특이값과 방향 성분으로 분해합니다.

# 가장 작은 특이값에 해당하는 오른쪽 특이벡터를 가져옵니다.
null_direction = Vh[-1, :].reshape(2, 1)  # 마지막 행은 영공간 방향에 가까운 벡터입니다.

# SVD로 찾은 영공간 방향을 A에 곱해봅니다.
AX_svd = A @ null_direction  # 거의 0에 가까운 값이 나오면 영공간 방향입니다.

# 결과를 출력합니다.
print_matrix("A", A)  # 영공간을 구할 행렬입니다.
print_matrix("X = [-2t, t], t=3", X_null)  # 강의자료에서 제시한 형태의 벡터입니다.
print_matrix("A @ X", AX)  # 0벡터가 나오는지 확인합니다.
print_matrix("SVD로 찾은 영공간 방향", null_direction)  # PyTorch로 찾은 영공간 방향입니다.
print_matrix("A @ null_direction", AX_svd)  # 거의 0에 가까운지 확인합니다.



A =
tensor([[ 1.,  2.],
        [-2., -4.]])
shape: (2, 2)

X = [-2t, t], t=3 =
tensor([[-6.],
        [ 3.]])
shape: (2, 1)

A @ X =
tensor([[0.],
        [0.]])
shape: (2, 1)

SVD로 찾은 영공간 방향 =
tensor([[ 0.8944],
        [-0.4472]])
shape: (2, 1)

A @ null_direction =
tensor([[-0.0000],
        [ 0.0000]])
shape: (2, 1)


## 12. 머신러닝과 딥러닝에서 행렬이 사용되는 위치

행렬은 단순한 수학 개념이 아니라 딥러닝 모델의 핵심 계산 구조입니다. 예를 들어 완전연결층(Dense Layer 또는 Linear Layer)은 다음 계산을 수행합니다.

\[
Y = XW + b
\]

여기서 `X`는 입력 데이터 행렬, `W`는 가중치 행렬, `b`는 편향 벡터, `Y`는 출력 행렬입니다. 아래 코드는 PyTorch의 `torch.nn.Linear` 없이 행렬 곱셈만으로 선형층 계산을 직접 구현합니다.


In [23]:
# 입력 데이터 X를 생성합니다. 4개의 데이터가 있고 각 데이터는 3개의 특성을 가집니다.
X = torch.tensor([[1.0, 2.0, 3.0],  # 첫 번째 샘플입니다.
                  [4.0, 5.0, 6.0],  # 두 번째 샘플입니다.
                  [7.0, 8.0, 9.0],  # 세 번째 샘플입니다.
                  [2.0, 4.0, 6.0]], # 네 번째 샘플입니다.
                 dtype=torch.float32)  # 딥러닝 계산을 위해 실수형으로 저장합니다.

# 가중치 행렬 W를 생성합니다. 입력 특성 3개를 출력 특성 2개로 바꾸기 위해 크기는 3x2입니다.
W = torch.tensor([[ 0.1,  0.2],  # 첫 번째 입력 특성에 대한 두 출력 뉴런의 가중치입니다.
                  [ 0.3, -0.1],  # 두 번째 입력 특성에 대한 두 출력 뉴런의 가중치입니다.
                  [-0.2,  0.4]], # 세 번째 입력 특성에 대한 두 출력 뉴런의 가중치입니다.
                 dtype=torch.float32)  # 행렬 곱셈을 위해 실수형으로 저장합니다.

# 편향 벡터 b를 생성합니다. 출력 특성이 2개이므로 편향도 2개입니다.
b = torch.tensor([0.5, -0.5], dtype=torch.float32)  # 각 출력 뉴런에 더해질 편향입니다.

# 선형층 계산 Y = XW + b를 수행합니다.
Y = X @ W + b  # 행렬 곱셈 후 PyTorch 브로드캐스팅으로 편향이 각 행에 더해집니다.

# 입력 행렬 X를 출력합니다.
print_matrix("X: 입력 데이터 행렬", X)  # 4개의 샘플과 3개의 특성을 확인합니다.

# 가중치 행렬 W를 출력합니다.
print_matrix("W: 가중치 행렬", W)  # 입력 3개를 출력 2개로 변환하는 가중치입니다.

# 편향 벡터 b를 출력합니다.
print_matrix("b: 편향 벡터", b)  # 각 출력 뉴런에 더해질 값입니다.

# 선형층 출력 Y를 출력합니다.
print_matrix("Y = XW + b", Y)  # 딥러닝 선형층의 출력 결과입니다.



X: 입력 데이터 행렬 =
tensor([[1., 2., 3.],
        [4., 5., 6.],
        [7., 8., 9.],
        [2., 4., 6.]])
shape: (4, 3)

W: 가중치 행렬 =
tensor([[ 0.1000,  0.2000],
        [ 0.3000, -0.1000],
        [-0.2000,  0.4000]])
shape: (3, 2)

b: 편향 벡터 =
tensor([ 0.5000, -0.5000])
shape: (2,)

Y = XW + b =
tensor([[0.6000, 0.7000],
        [1.2000, 2.2000],
        [1.8000, 3.7000],
        [0.7000, 1.9000]])
shape: (4, 2)
